In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/creditcard.csv")
print(df.shape)

(284807, 31)


In [2]:
df['Class'].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

In [2]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])

In [3]:
df = df.drop(['Amount'], axis=1)

In [4]:
df = df.drop(['Time'], axis=1)

In [5]:
fraud = df[df['Class'] == 1]
normal = df[df['Class'] == 0]

In [6]:
normal_sample = normal.sample(len(fraud))

In [7]:
df_balanced = pd.concat([fraud, normal_sample])

In [8]:
df_balanced = df_balanced.sample(frac=1, random_state=42)

In [9]:
X = df_balanced.drop('Class', axis=1)
y = df_balanced['Class']

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Initialize model
model = LogisticRegression()

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[ 85   2]
 [  9 101]]
              precision    recall  f1-score   support

           0       0.90      0.98      0.94        87
           1       0.98      0.92      0.95       110

    accuracy                           0.94       197
   macro avg       0.94      0.95      0.94       197
weighted avg       0.95      0.94      0.94       197



In [14]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[ 82   5]
 [  8 102]]
              precision    recall  f1-score   support

           0       0.91      0.94      0.93        87
           1       0.95      0.93      0.94       110

    accuracy                           0.93       197
   macro avg       0.93      0.93      0.93       197
weighted avg       0.93      0.93      0.93       197



In [15]:
from sklearn.metrics import classification_report, confusion_matrix

y_probs = model.predict_proba(X_test)[:, 1]

threshold = 0.3
y_pred = (y_probs > threshold).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[ 72  15]
 [  6 104]]
              precision    recall  f1-score   support

           0       0.92      0.83      0.87        87
           1       0.87      0.95      0.91       110

    accuracy                           0.89       197
   macro avg       0.90      0.89      0.89       197
weighted avg       0.90      0.89      0.89       197



In [11]:
X_full = df.drop('Class', axis=1)
y_full = df['Class']

In [12]:
from sklearn.model_selection import train_test_split

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full,
    y_full,
    test_size=0.2,
    random_state=42,
    stratify=y_full   
)

In [13]:
from sklearn.ensemble import RandomForestClassifier

model_full = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',   # KEY FIX FOR IMBALANCE
    random_state=42,
    n_jobs=-1                  # use all CPU cores
)

model_full.fit(X_train_full, y_train_full)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [14]:
y_pred_full = model_full.predict(X_test_full)

In [15]:
from sklearn.metrics import classification_report, confusion_matrix

print("CONFUSION MATRIX (FULL DATASET):")
print(confusion_matrix(y_test_full, y_pred_full))

print("\nCLASSIFICATION REPORT (FULL DATASET):")
print(classification_report(y_test_full, y_pred_full))

CONFUSION MATRIX (FULL DATASET):
[[56861     3]
 [   24    74]]

CLASSIFICATION REPORT (FULL DATASET):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.96      0.76      0.85        98

    accuracy                           1.00     56962
   macro avg       0.98      0.88      0.92     56962
weighted avg       1.00      1.00      1.00     56962



In [16]:
y_probs = model_full.predict_proba(X_test_full)[:, 1]

threshold = 0.3   # try 0.3 first
y_pred_thresh = (y_probs > threshold).astype(int)

from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test_full, y_pred_thresh))
print(classification_report(y_test_full, y_pred_thresh))

[[56858     6]
 [   19    79]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.93      0.81      0.86        98

    accuracy                           1.00     56962
   macro avg       0.96      0.90      0.93     56962
weighted avg       1.00      1.00      1.00     56962



In [17]:
y_probs = model_full.predict_proba(X_test_full)[:, 1]
threshold = 0.2
y_pred_thresh = (y_probs > threshold).astype(int)

from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test_full, y_pred_thresh))
print(classification_report(y_test_full, y_pred_thresh))

[[56851    13]
 [   14    84]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.87      0.86      0.86        98

    accuracy                           1.00     56962
   macro avg       0.93      0.93      0.93     56962
weighted avg       1.00      1.00      1.00     56962



In [18]:
y_probs = model_full.predict_proba(X_test_full)[:, 1]
threshold = 0.1
y_pred_thresh = (y_probs > threshold).astype(int)

from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test_full, y_pred_thresh))
print(classification_report(y_test_full, y_pred_thresh))

[[56837    27]
 [   12    86]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.76      0.88      0.82        98

    accuracy                           1.00     56962
   macro avg       0.88      0.94      0.91     56962
weighted avg       1.00      1.00      1.00     56962



In [19]:
y_probs = model_full.predict_proba(X_test_full)[:, 1]
threshold = 0.05
y_pred_thresh = (y_probs > threshold).astype(int)

from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test_full, y_pred_thresh))
print(classification_report(y_test_full, y_pred_thresh))

[[56815    49]
 [   10    88]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.64      0.90      0.75        98

    accuracy                           1.00     56962
   macro avg       0.82      0.95      0.87     56962
weighted avg       1.00      1.00      1.00     56962



In [20]:
import pickle
import os

# Create models folder if not exists
os.makedirs("../models", exist_ok=True)

# Save model
with open("../models/fraud_model.pkl", "wb") as f:
    pickle.dump(model_full, f)

# Save scaler (IMPORTANT)
with open("../models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Save threshold
with open("../models/threshold.pkl", "wb") as f:
    pickle.dump(0.2, f)

print("✅ Model, scaler, and threshold saved successfully!")

✅ Model, scaler, and threshold saved successfully!


In [21]:
with open("../models/fraud_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

print("Model loaded successfully!")

Model loaded successfully!


In [22]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train_full, y_train_full)

print("Before SMOTE:", y_train_full.value_counts())
print("After SMOTE:", y_train_smote.value_counts())

Before SMOTE: Class
0    227451
1       394
Name: count, dtype: int64
After SMOTE: Class
0    227451
1    227451
Name: count, dtype: int64


In [23]:
from sklearn.ensemble import RandomForestClassifier

model_smote = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model_smote.fit(X_train_smote, y_train_smote)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [24]:
y_probs_smote = model_smote.predict_proba(X_test_full)[:, 1]

In [25]:
threshold = 0.2

y_pred_smote = (y_probs_smote > threshold).astype(int)

In [26]:
from sklearn.metrics import confusion_matrix, classification_report

print("CONFUSION MATRIX (SMOTE):")
print(confusion_matrix(y_test_full, y_pred_smote))

print("\nCLASSIFICATION REPORT (SMOTE):")
print(classification_report(y_test_full, y_pred_smote))

CONFUSION MATRIX (SMOTE):
[[56805    59]
 [   10    88]]

CLASSIFICATION REPORT (SMOTE):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.60      0.90      0.72        98

    accuracy                           1.00     56962
   macro avg       0.80      0.95      0.86     56962
weighted avg       1.00      1.00      1.00     56962



In [27]:
for t in [0.3, 0.4, 0.5]:
    y_pred = (y_probs_smote > t).astype(int)
    
    print(f"\nThreshold: {t}")
    print(confusion_matrix(y_test_full, y_pred))


Threshold: 0.3
[[56831    33]
 [   11    87]]

Threshold: 0.4
[[56846    18]
 [   14    84]]

Threshold: 0.5
[[56852    12]
 [   17    81]]


In [28]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, classification_report

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train_full, y_train_full)

# Probabilities
gb_probs = gb.predict_proba(X_test_full)[:, 1]

# Use same threshold as baseline
threshold = 0.2
gb_pred = (gb_probs > threshold).astype(int)

print("GRADIENT BOOSTING @ 0.2")
print(confusion_matrix(y_test_full, gb_pred))
print(classification_report(y_test_full, gb_pred))

GRADIENT BOOSTING @ 0.2
[[56838    26]
 [   71    27]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.51      0.28      0.36        98

    accuracy                           1.00     56962
   macro avg       0.75      0.64      0.68     56962
weighted avg       1.00      1.00      1.00     56962



In [29]:
from xgboost import XGBClassifier

# scale_pos_weight ≈ (negatives / positives) in training set
neg, pos = (y_train_full == 0).sum(), (y_train_full == 1).sum()
scale_pos_weight = neg / pos

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,  # KEY for imbalance
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

xgb.fit(X_train_full, y_train_full)

# Probabilities
xgb_probs = xgb.predict_proba(X_test_full)[:, 1]

# Same threshold
threshold = 0.2
xgb_pred = (xgb_probs > threshold).astype(int)

from sklearn.metrics import confusion_matrix, classification_report
print("XGBOOST @ 0.2")
print(confusion_matrix(y_test_full, xgb_pred))
print(classification_report(y_test_full, xgb_pred))

XGBOOST @ 0.2
[[56833    31]
 [   12    86]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.74      0.88      0.80        98

    accuracy                           1.00     56962
   macro avg       0.87      0.94      0.90     56962
weighted avg       1.00      1.00      1.00     56962



In [30]:
for t in [0.2, 0.25, 0.3]:
    pred = (xgb_probs > t).astype(int)
    print(f"\nThreshold: {t}")
    print(confusion_matrix(y_test_full, pred))


Threshold: 0.2
[[56833    31]
 [   12    86]]

Threshold: 0.25
[[56836    28]
 [   14    84]]

Threshold: 0.3
[[56841    23]
 [   15    83]]
